# pttai example

The schema-free `>` DSL on a **real** OpenAI model, showcasing the seven shipped features.

**API key:** set `OPENAI_API_KEY` in your environment, or drop it in a `.env` at the repo
root (`OPENAI_API_KEY=sk-...`) — the setup cell calls `load_dotenv()`.

Each cell builds its own nodes, so you can run them in any order.

## 0. Setup

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from pttai import AgentNode, AgenticGraph, AgenticState, fanout
from pttai.validation import GraphValidationError

load_dotenv()  # reads OPENAI_API_KEY from a .env at the repo root, if present

# One shared model — ChatOpenAI is stateless per call, so every node can reuse it.
llm = ChatOpenAI(model="gpt-5.4-nano")

print("ready — pttai imported, llm =", llm.model_name)

## 1. Parallel fan-out + join  ·  token usage  ·  prompt caching

`start` fans out to two workers that run **concurrently**; `combine` is a deferred join that runs **once** after both. No `state=` → standard `AgenticState`; `message=` is the input shorthand; `prompt_cache=True` opts into OpenAI prompt caching; `out["token"]` reports per-model usage.

In [ ]:
start = AgentNode(name="start", llm=llm,
                  node_prompt="Restate the user's request in one short line.")
worker_a = AgentNode(name="worker_a", llm=llm,
                     node_prompt="List two PROS of the topic in the conversation. Be brief.")
worker_b = AgentNode(name="worker_b", llm=llm,
                     node_prompt="List two CONS of the topic in the conversation. Be brief.")
combine = AgentNode(name="combine", llm=llm,
                    node_prompt="Weigh the pros and cons above into a one-sentence verdict.")

start > fanout(worker_a, worker_b) > combine
# equivalent: start > [worker_a, worker_b] > combine

graph = AgenticGraph(start_node=start, end_nodes={combine}, prompt_cache=True)
out = graph.invoke(message="Should a small team adopt a monorepo?")

print("final verdict:", out["messages"][-1].content)
print("\ntrace (both branches ran, join ran once):")
for line in out["log"]:
    print("  -", line)
print("\ntoken usage per model:")
for model, usage in out["token"].items():
    print(f"  {model}: {usage.get('total_tokens')} total")

## 2. Map-reduce

`summarize.map("docs")` fans the worker out via LangGraph `Send`, once per item of `state["docs"]`, all in parallel; `reduce` joins once after. `docs` is a real list data-channel, so it lives on a small state extending the standard one — the one place a schema is still earned.

In [ ]:
class MapState(AgenticState):
    docs: list


dispatch = AgentNode(name="dispatch", llm=llm,
                     node_prompt="Acknowledge that you'll summarize the documents.")
summarize = AgentNode(name="summarize", llm=llm,
                      node_prompt="Summarize the message into a single short sentence.")
reduce = AgentNode(name="reduce", llm=llm,
                   node_prompt="Combine the per-document summaries above into one digest.")

dispatch > summarize.map("docs") > reduce

graph = AgenticGraph(state=MapState, start_node=dispatch, end_nodes={reduce})
out = graph.invoke(
    message="summarize these",
    docs=[
        "LangGraph models agent workflows as a stateful graph of nodes and edges.",
        "Keras became the default high-level API for TensorFlow by winning on ergonomics.",
        "pttai compiles a Pythonic '>' DSL down to a native LangGraph StateGraph.",
    ],
)
print("final digest:", out["messages"][-1].content)

## 3. Schema-free typed multi-key IO

No custom `TypedDict`: `topic` (read, never written) auto-registers as an input; `sentiment`/`score` (written) auto-register too. `writes` as a `dict[str, type]` gives **typed** structured output, so `score` comes back a native `int`.

In [ ]:
classify = AgentNode(
    name="classify",
    llm=llm,
    node_prompt="Rate the {topic} discussion. Return a sentiment and a 1-10 score.",
    reads=["messages", "topic"],
    writes={"sentiment": str, "score": int},
)

graph = AgenticGraph(start_node=classify, end_nodes={classify})   # no state=
out = graph.invoke(message="Honestly I loved it, fast and clean.", topic="product")
print(f"sentiment={out['sentiment']!r}  score={out['score']!r}  (score is a real {type(out['score']).__name__})")

## 4. `graph.summary()` — the Keras-style table

node · type · reads · writes · available keys. (No model calls.)

In [ ]:
s_start = AgentNode(name="start", llm=llm)
s_a = AgentNode(name="worker_a", llm=llm)
s_b = AgentNode(name="worker_b", llm=llm)
s_combine = AgentNode(name="combine", llm=llm)
s_start > fanout(s_a, s_b) > s_combine

AgenticGraph(start_node=s_start, end_nodes={s_combine}).summary()

## 5. Build-time validation

Schema-free, but the guardrail still fires: `early` reads `summary`, which `late` writes **downstream** — read-before-write, caught at build on the default state. (No model calls.)

In [ ]:
early = AgentNode(name="early", llm=llm, reads=["summary"])
late = AgentNode(name="late", llm=llm, output_field="summary")
early > late

try:
    AgenticGraph(start_node=early, end_nodes={late})   # no state=
    print("no error (unexpected)")
except GraphValidationError as e:
    print("caught GraphValidationError at build time:\n")
    print(e)